# 05 — Figures and tables

Part 1 loads the saved arrays only (run notebooks 02–04 first) and writes the result figures to `results/figures/` (300 dpi) and the results table to `results/table.csv`. Result figures plot **error rate = 1 − accuracy**, mean over splits.

Part 2 makes explanatory figures (eigen-images, reconstructions, 2-D projections, per-object errors). These need the dataset and refit on split 0 or at the validation-selected values; they run no new experiments.

The sample image grid (`samples.png`) is made in notebook 01.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter, MultipleLocator, PercentFormatter

from src.config import RESULTS_DIR

ARRAYS = RESULTS_DIR / "arrays"
FIGURES = RESULTS_DIR / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", "Mean of empty slice", RuntimeWarning)          # min-Mahal is nan above the cap
warnings.filterwarnings("ignore", "Degrees of freedom <= 0", RuntimeWarning)

# Colour = classifier identity, fixed across the result figures; marker is the second cue (print / colour-blind)
STYLE = {
    "1-NN":       dict(color="#2a78d6", marker="o"),
    "min-Euclid": dict(color="#eb6834", marker="s"),
    "min-Mahal":  dict(color="#1baf7a", marker="^"),
}
EPS_RAMP = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"]   # ordinal: small -> large eps
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
# diverging blue <-> red with a neutral grey midpoint, for signed images (eigen-images)
DIVERGING = LinearSegmentedColormap.from_list("div", ["#0d366b", "#2a78d6", "#f0efec", "#e34948", "#8f1f1f"])

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "font.size": 9, "axes.titlesize": 9, "axes.titleweight": "normal", "axes.titlecolor": INK,
    "figure.titlesize": 10, "figure.titleweight": "bold",
    "axes.labelcolor": INK2, "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "grid.linestyle": "-",
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "lines.linewidth": 1.6, "lines.markersize": 5, "legend.frameon": False, "legend.fontsize": 8,
    "legend.labelcolor": INK, "savefig.dpi": 300, "savefig.bbox": "tight",
})


def pct(ax, decimals=0):
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=decimals))


def save(fig, name):
    fig.savefig(FIGURES / name)
    plt.show()


def image_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)

In [ ]:
pca = np.load(ARRAYS / "pca_acc.npz")
base = np.load(ARRAYS / "baseline.npz")
eig = np.load(ARRAYS / "eigvals.npy")
lda = np.load(ARRAYS / "lda_acc.npz")
lda_m = np.load(ARRAYS / "lda_m_sweep.npz")
sel = np.load(ARRAYS / "selected.npz")
e4 = np.load(ARRAYS / "trainsize_acc.npz")
e5 = np.load(ARRAYS / "random_split_acc.npz")
e6 = np.load(ARRAYS / "reg_mahal_acc.npz")
e6_sel = np.load(ARRAYS / "reg_mahal_selected.npz")
names = [str(n) for n in pca["classifiers"]]
d_list = pca["d_list"]
mark_at = [np.flatnonzero(d_list == d)[0] for d in (1, 2, 5, 10, 20, 50, 100, 200)]

# Part 1: result figures

## Eigenspectrum (E3)

In [ ]:
k = np.arange(1, eig.shape[1] + 1)
cum = np.cumsum(eig, axis=1) / eig.sum(axis=1, keepdims=True)
blue = STYLE["1-NN"]["color"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 2.9))
ax1.fill_between(k, eig.min(axis=0), eig.max(axis=0), color=blue, alpha=0.2, lw=0)
ax1.semilogy(k, eig.mean(axis=0), color=blue)
ax1.set(xlabel="component index", ylabel="eigenvalue (log scale)", title="(a) PCA eigenvalues", xlim=(0, k[-1] + 1))
ax2.plot(k, cum.mean(axis=0), color=blue)
for p, y_text in ((0.90, 0.60), (0.95, 0.35), (0.99, 0.60)):   # staggered: 43 and 72 are close
    d_p = int(np.median((cum < p).sum(axis=1) + 1))
    ax2.plot([d_p, d_p], [0, p], color=AXIS, lw=0.8, zorder=0)
    ax2.plot(d_p, p, "o", color=blue, mec="white", mew=1.5, ms=7)
    ax2.annotate(f"{p:.0%}\nd = {d_p}", (d_p, y_text), ha="center", va="center", color=INK, fontsize=8,
                 bbox=dict(fc="white", ec="none", pad=1.5))
ax2.set(xlabel="number of components d", ylabel="cumulative variance", title="(b) Variance retained",
        ylim=(0, 1.02), xlim=(0, k[-1] + 1))
pct(ax2)
d80 = int(np.median((cum < 0.8).sum(axis=1) + 1))
fig.suptitle(f"Variance is concentrated: {d80} of {k[-1]} components hold 80%")
fig.tight_layout()
save(fig, "eigenspectrum.png")

## PCA error rate vs d (E0, E1)

(a) full range; (b) zoom on d ≥ 10 where the classifiers separate. Shaded band = ± 1 std over the 6 splits. Open markers = validation-selected d (median over splits). Dashed lines = raw-pixel baselines (no reduction).

In [ ]:
err, sd = 1 - np.nanmean(pca["acc"], axis=0), np.nanstd(pca["acc"], axis=0)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 3.3), gridspec_kw=dict(width_ratios=[1, 1.3]))
for ax in (ax1, ax2):
    for i, n in enumerate(names):
        s = STYLE[n]
        ok = ~np.isnan(err[i])
        ax.fill_between(d_list[ok], (err[i] - sd[i])[ok], (err[i] + sd[i])[ok], color=s["color"], alpha=0.15, lw=0)
        ax.plot(d_list[ok], err[i][ok], color=s["color"], marker=s["marker"],
                markevery=[m for m in mark_at if ok[m]], label=n)
    for i, n in enumerate(names[:2]):
        ax.axhline(1 - base["acc"][:, i].mean(), color=STYLE[n]["color"], lw=1, ls="--",
                   label=f"{n}, raw pixels (n = 1024)")
    ax.axvline(220, color=MUTED, lw=0.8, zorder=0)
    ax.set(xscale="log", xlabel="PCA dimension d (log scale)")
for i, n in enumerate(names):
    s = STYLE[n]
    j = np.argmin(np.abs(d_list - np.median(sel["pca_d"][:, i])))
    ax2.plot(d_list[j], err[i, j], s["marker"], ms=9, mfc="white", mec=s["color"], mew=1.8, zorder=5)
ax2.plot([], [], "o", ms=8, mfc="white", mec=INK2, mew=1.5, label="validation-selected d")
ax1.set(ylabel="test error rate", title="(a) All dimensions", ylim=(0, 0.7))
ax2.set(title="(b) Zoom: d ≥ 10", xlim=(10, 260), ylim=(0, 0.15))
pct(ax1)
pct(ax2)
ax2.annotate("N − c = 220: min-Mahal cap\n(min-Mahal ↑ 43% at the cap)", (220, 0.147), xytext=(-5, 0),
             textcoords="offset points", ha="right", va="top", color=INK2, fontsize=7.5)
h, l = ax2.get_legend_handles_labels()
fig.legend(h, l, loc="upper center", bbox_to_anchor=(0.5, 0.02), ncol=3)
fig.suptitle("PCA + 1-NN at d ≈ 16 beats 1-NN on all 1024 raw pixels")
fig.tight_layout()
save(fig, "e1_pca_error_vs_d.png")

## PCA → LDA (E2)

(a) At m = N − c each split behaves very differently, so the six splits are drawn individually. In LDA space min-Mahal and min-Euclid give identical predictions ($W^T S_w W = I$), so they are one line. (b) Error at d = 19 against the intermediate PCA dimension m, log scale.

In [ ]:
dl = lda["d_list"]
orange, blue = STYLE["min-Euclid"]["color"], STYLE["1-NN"]["color"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 3.1))
per_split = 1 - lda["acc"][:, 1, :]                      # min-Euclid = min-Mahal
for k_, e in enumerate(per_split):
    ax1.plot(dl, e, color=orange, lw=0.9, alpha=0.75, label="each of the 6 splits" if k_ == 0 else None)
ax1.plot(dl, per_split.mean(axis=0), color=orange, lw=2.4, marker="s", markevery=3, label="mean, min-Euclid = min-Mahal")
ax1.plot(dl, 1 - lda["acc"][:, 0, :].mean(axis=0), color=blue, lw=1.2, marker="o", markevery=(1, 3), label="mean, 1-NN")
ax1.set(xlabel="LDA dimension d", ylabel="test error rate", title="(a) m = N − c = 220: unstable across splits",
        xticks=[1, 4, 7, 10, 13, 16, 19], ylim=(0, 0.9))
pct(ax1)
ax1.legend(loc="upper right")

m_list = lda_m["m_list"]
acc_m = lda_m["acc"][:, :, :, -1]   # d = c - 1 = 19
shown = {"1-NN": "1-NN", "min-Euclid": "min-Euclid = min-Mahal"}
for i, n in enumerate(names[:2]):
    s = STYLE[n]
    e = 1 - acc_m[:, :, i]
    ax2.errorbar(m_list, e.mean(axis=1), yerr=e.std(axis=1), color=s["color"], marker=s["marker"],
                 capsize=2, elinewidth=0.8, label=shown[n])
ax2.axvline(220, color=MUTED, lw=0.8, zorder=0)
ax2.annotate("N − c", (220, 0.012), xytext=(-4, 0), textcoords="offset points", ha="right", color=INK2, fontsize=8)
ax2.set(yscale="log", xlabel="intermediate PCA dimension m", ylabel="test error rate at d = 19 (log scale)",
        title="(b) Small m works; m = N − c fails", ylim=(0.01, 1))
pct(ax2)
ax2.legend(loc="upper left")
fig.suptitle("LDA overfits when the intermediate PCA keeps all N − c dimensions")
fig.tight_layout()
save(fig, "e2_lda_error.png")

## Training size (E4)

Test error at validation-selected d (PCA) and (m, d) (PCA → LDA). Error bars = ± 1 std over splits. Solid/filled = PCA, dashed/open = PCA → LDA.

In [ ]:
steps = e4["steps"]
per_class = 72 // steps
fig, ax = plt.subplots(figsize=(7.2, 3.4))
for i, n in enumerate(names):
    s = STYLE[n]
    for method, ls in (("pca", "-"), ("lda", "--")):
        if method == "lda" and n == "min-Mahal":
            continue   # identical to PCA -> LDA min-Euclid
        acc = [e4[f"{method}_test_{st}"][:, i] for st in steps]
        label = f"PCA {n}" if method == "pca" else ("PCA→LDA min-Euclid = min-Mahal" if n == "min-Euclid" else f"PCA→LDA {n}")
        ax.errorbar(per_class, [1 - a.mean() for a in acc], yerr=[a.std() for a in acc], color=s["color"],
                    marker=s["marker"], ls=ls, mfc="white" if method == "lda" else s["color"],
                    capsize=2, elinewidth=0.8, label=label)
ax.set(xlabel="training images per class (pose spacing 60°, 45°, 30°, 20°)", ylabel="test error rate",
       xticks=per_class, ylim=(0, None))
pct(ax)
ax.legend(ncol=2)
fig.suptitle("More training poses help every method except min-Euclid, which levels off near 9%")
fig.tight_layout()
save(fig, "e4_trainsize.png")

## Random vs pose-spaced split (E5)

Extra error of a random split over the pose-spaced split of the same size (positive = random is worse), mean over 6 seeds / splits.

In [ ]:
pts = FuncFormatter(lambda v, _: f"{100 * v:+.0f} pts" if v else "0")
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0), sharey=True)
for ax, n_pc in zip(axes, (12, 36)):
    gap = np.nanmean(e5[f"spaced_{n_pc}"], axis=0) - np.nanmean(e5[f"random_{n_pc}"], axis=0)
    ax.axhline(0, color=AXIS, lw=1, zorder=0)
    for i, n in enumerate(names):
        ok = ~np.isnan(gap[i])
        s = STYLE[n]
        ax.plot(e5["d_list"][ok], gap[i][ok], color=s["color"], marker=s["marker"],
                markevery=[m for m in mark_at if ok[m]], label=n)
    ax.set(xscale="log", xlabel="PCA dimension d (log scale)", title=f"{n_pc} training images per class",
           xlim=(2, 200), ylim=(-0.03, 0.12))   # stop at 200: min-Mahal collapses at its cap in both splits
    ax.yaxis.set_major_locator(MultipleLocator(0.02))
    ax.yaxis.set_major_formatter(pts)
axes[0].set_ylabel("error: random − pose-spaced")
axes[0].legend(loc="upper left")
fig.suptitle("Random splits leave gaps in pose coverage and are worse at almost every d")
fig.tight_layout()
save(fig, "e5_random_vs_spaced.png")

## Regularised Mahalanobis (E6)

Zoomed to d ≥ 40: below that all ε give the same curve.

In [ ]:
eps_list = e6["eps_list"]
err6 = 1 - np.nanmean(e6["acc"], axis=0)
fig, ax = plt.subplots(figsize=(7.2, 3.2))
for i, (e, c) in enumerate(zip(eps_list, EPS_RAMP)):
    ok = ~np.isnan(err6[i])
    ax.plot(d_list[ok], err6[i][ok], color=c, lw=2 if e == 0 else 1.6,
            label=f"ε = {e:g}" + (" (unregularised)" if e == 0 else ""))
ax.axvline(220, color=MUTED, lw=0.8, zorder=0)
ax.annotate("unregularised ↑ 43%\nat N − c = 220", (190, 0.148), xytext=(-5, 0), textcoords="offset points",
            ha="right", va="top", color=INK2, fontsize=8)
ax.set(xscale="log", xlabel="PCA dimension d (log scale)", ylabel="test error rate", xlim=(40, 260), ylim=(0, 0.15))
ax.set_xticks([40, 60, 100, 150, 220])
ax.set_xticklabels(["40", "60", "100", "150", "220"])
ax.minorticks_off()
pct(ax)
ax.legend(loc="upper left", ncol=2)
fig.suptitle(r"Regularising $\Sigma_w + \epsilon\,(\mathrm{tr}\,\Sigma_w / d)\,I$ removes the collapse near the cap")
fig.tight_layout()
save(fig, "e6_reg_mahal.png")

## Results table

Test accuracy (mean ± std over splits) at the validation-selected value, plus the best point on the test curve for reference (**oracle**: chosen with test data, not a valid result).

In [ ]:
def spread(a):
    a = np.asarray(a)
    return f"{a.min():g}" if a.min() == a.max() else f"{np.median(a):g} [{a.min():g}–{a.max():g}]"

rows = []
for i, n in enumerate(names[:2]):
    a = base["acc"][:, i]
    rows.append(("Raw pixels", n, "n = 1024", a.mean(), a.std(), np.nan))
best_pca = np.nanmax(np.nanmean(pca["acc"], axis=0), axis=1)
for i, n in enumerate(names):
    a = sel["pca_test"][:, i]
    rows.append(("PCA", n, f"d = {spread(sel['pca_d'][:, i])}", a.mean(), a.std(), best_pca[i]))
for i, n in enumerate(names):
    a = lda["acc"][:, i, -1]
    rows.append(("PCA(N − c) → LDA", n, "fixed: m = 220, d = 19", a.mean(), a.std(), lda["acc"][:, i].mean(axis=0).max()))
best_lda = lda_m["acc"].mean(axis=1).max(axis=(0, 2))
for i, n in enumerate(names):
    a = sel["lda_test"][:, i]
    rows.append(("PCA(m) → LDA", n, f"m = {spread(sel['lda_m'][:, i])}, d = {spread(sel['lda_d'][:, i])}",
                 a.mean(), a.std(), best_lda[i]))
a = e6_sel["test"]
eps_chosen = ", ".join(f"{e:g}" for e in sorted(set(e6_sel["eps"])))
rows.append(("PCA, regularised", "min-Mahal", f"ε ∈ {{{eps_chosen}}}, d = {spread(e6_sel['d'])}",
             a.mean(), a.std(), np.nanmax(np.nanmean(e6["acc"], axis=0))))

table = pd.DataFrame(rows, columns=["method", "classifier", "selected by validation",
                                    "test acc mean", "test acc std", "best on test curve (oracle)"])
table.to_csv(RESULTS_DIR / "table.csv", index=False, float_format="%.4f")
view = table[["method", "classifier", "selected by validation"]].copy()
view["test accuracy"] = [f"{m:.1%} ± {s:.1%}" for m, s in zip(table["test acc mean"], table["test acc std"])]
view["oracle"] = ["–" if np.isnan(b) else f"{b:.1%}" for b in table["best on test curve (oracle)"]]
view

# Part 2: explanatory figures

These refit PCA / LDA on split 0 (training data only) to show what the methods do. They need the dataset.

In [ ]:
from src.classifiers import nn_predict
from src.config import IMG_SIZE
from src.data import load_coil20, make_split
from src.reduction import fit_lda, fit_pca, pca_project

X, y, pose = load_coil20()
X_raw, _, _ = load_coil20(normalise=False)
tr0, te0 = make_split(pose, 0)
m0, V0, lam0 = fit_pca(X[tr0])
N0, c0 = len(tr0), len(np.unique(y[tr0]))


def show(ax, v, **kw):
    ax.imshow(v.reshape(IMG_SIZE, IMG_SIZE), **kw)
    image_axes(ax)

## Eigen-images

The training mean and the first principal directions, each reshaped to 32 × 32. Red = positive, blue = negative weight. Percentages are the share of variance.

In [ ]:
n_pc = 7
fig, axes = plt.subplots(1, n_pc + 1, figsize=(7.2, 1.55))
show(axes[0], m0, cmap="gray")
axes[0].set_title("mean", fontsize=8)
share = lam0 / lam0.sum()
for j in range(n_pc):
    v = V0[:, j]
    lim = np.abs(v).max()
    show(axes[j + 1], v, cmap=DIVERGING, vmin=-lim, vmax=lim)
    axes[j + 1].set_title(f"PC {j + 1} ({share[j]:.0%})", fontsize=8)
fig.suptitle("Leading PCA directions capture coarse shape and brightness layout, not object detail", y=1.0)
save(fig, "pca_eigenimages.png")

## Reconstructions

Test images (not used to fit PCA) rebuilt from the first d components: $\hat{x} = m + V_d V_d^T (x - m)$.

In [ ]:
d_show = [5, 20, 50, 200]
examples = [(0, 1), (4, 1), (18, 1)]   # (object index, pose): objects 1, 5, 19 at 5°, all test images on split 0
fig, axes = plt.subplots(len(examples), len(d_show) + 1, figsize=(5.2, 3.4))
for r, (obj, p) in enumerate(examples):
    i = np.flatnonzero((y == obj) & (pose == p))[0]
    assert i in te0
    lim = np.abs(X[i]).max()
    show(axes[r, 0], X[i], cmap="gray", vmin=-lim, vmax=lim)
    for c, d in enumerate(d_show):
        x_hat = m0 + V0[:, :d] @ pca_project(X[i:i + 1], m0, V0, d)[0]
        show(axes[r, c + 1], x_hat, cmap="gray", vmin=-lim, vmax=lim)
    axes[r, 0].set_ylabel(f"obj {obj + 1}", fontsize=8, color=INK2)
    axes[r, 0].yaxis.set_visible(True)
axes[0, 0].set_title("original", fontsize=8)
for c, d in enumerate(d_show):
    v = lam0[:d].sum() / lam0.sum()
    axes[0, c + 1].set_title(f"d = {d} ({100 * v:.1f}%)", fontsize=8)
fig.suptitle("Coarse shape appears by d ≈ 20; text and fine detail need d ≥ 50", y=1.02)
fig.tight_layout()
save(fig, "pca_reconstruction.png")

## 2-D projections: PCA vs LDA

Split 0. Three often-confused objects are highlighted; the other 17 are grey. Filled = training, open = test. Only the first two of the d dimensions are shown, so overlap here does not mean the classifier fails.

In [ ]:
focus = {4: "obj 5", 6: "obj 7", 18: "obj 19"}   # most-confused objects (see per-object figure)
colours = ["#2a78d6", "#eb6834", "#1baf7a"]
markers = ["o", "s", "^"]
m_good = int(np.median(sel["lda_m"][:, 0]))


def lda_coords(mp):
    Ztr, Zte = pca_project(X[tr0], m0, V0, mp), pca_project(X[te0], m0, V0, mp)
    W, _ = fit_lda(Ztr, y[tr0])
    return (Ztr @ W)[:, :2], (Zte @ W)[:, :2]


panels = [("(a) PCA: PC 1 vs PC 2", (pca_project(X[tr0], m0, V0, 2), pca_project(X[te0], m0, V0, 2))),
          (f"(b) PCA(m = {m_good}) → LDA: LD 1 vs LD 2", lda_coords(m_good)),
          (f"(c) PCA(m = {N0 - c0}) → LDA: LD 1 vs LD 2", lda_coords(N0 - c0))]
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.8))
for ax, (title, (Ptr, Pte)) in zip(axes, panels):
    rest_tr, rest_te = ~np.isin(y[tr0], list(focus)), ~np.isin(y[te0], list(focus))
    ax.scatter(*Pte[rest_te].T, s=3, color=AXIS, lw=0)
    ax.scatter(*Ptr[rest_tr].T, s=6, color=MUTED, lw=0)
    for (obj, lab), col, mk in zip(focus.items(), colours, markers):
        ax.scatter(*Pte[y[te0] == obj].T, s=10, marker=mk, facecolor="white", edgecolor=col, lw=0.8)
        ax.scatter(*Ptr[y[tr0] == obj].T, s=30, marker=mk, color=col, edgecolor="white", lw=0.6, label=lab, zorder=5)
    ax.set_title(title, fontsize=8)
    ax.tick_params(labelbottom=False, labelleft=False, length=0)
axes[0].legend(loc="best", fontsize=7, handletextpad=0.2)
fig.suptitle("LDA tightens class clusters; with m = N − c each class's training images collapse to a point\n"
             "while its test images scatter (overfitting)", y=1.04)
fig.tight_layout()
save(fig, "projection_2d.png")

## Per-object error

PCA + 1-NN at the validation-selected d, all 6 splits (7200 test predictions). Bottom: the three most frequent confusions.

In [ ]:
conf = np.zeros((c0, c0), dtype=int)
for k_ in range(sel["pca_d"].shape[0]):
    tr, te = make_split(pose, k_)
    m, V, _ = fit_pca(X[tr])
    d = sel["pca_d"][k_, 0]
    np.add.at(conf, (y[te], nn_predict(pca_project(X[tr], m, V, d), y[tr], pca_project(X[te], m, V, d))), 1)
wrong = conf - np.diag(np.diag(conf))
obj_err = wrong.sum(axis=1) / conf.sum(axis=1)
pairs = np.dstack(np.unravel_index(np.argsort(-wrong.ravel())[:3], wrong.shape))[0]

fig = plt.figure(figsize=(7.2, 4.4))
gs = fig.add_gridspec(2, 6, height_ratios=[1.6, 1], hspace=0.55)
ax = fig.add_subplot(gs[0, :])
objs = np.arange(1, c0 + 1)
ax.bar(objs, obj_err, color=STYLE["1-NN"]["color"], width=0.7)
for o, e in zip(objs, obj_err):
    if e >= 0.05:
        ax.annotate(f"{e:.0%}", (o, e), xytext=(0, 2), textcoords="offset points", ha="center", color=INK, fontsize=7.5)
ax.set(xticks=objs, xlabel="object", ylabel="test error rate", xlim=(0.4, c0 + 0.6))
ax.grid(axis="x", visible=False)
ax.yaxis.set_major_locator(MultipleLocator(0.02))
pct(ax)
ax.set_title(f"{(obj_err == 0).sum()} of {c0} objects are never misclassified; errors concentrate on a few", fontsize=9)
pair_axes = []
for col, (a, b) in enumerate(pairs):
    row = []
    for side, obj in enumerate((a, b)):
        axi = fig.add_subplot(gs[1, 2 * col + side])
        i = np.flatnonzero((y == obj) & (pose == 0))[0]
        show(axi, X_raw[i], cmap="gray", vmin=0, vmax=1)
        axi.set_title(("true " if side == 0 else "→ predicted ") + f"{obj + 1}", fontsize=7.5, color=INK2)
        row.append(axi)
    pair_axes.append((row, wrong[a, b]))
fig.suptitle("Where PCA + 1-NN goes wrong")
fig.canvas.draw()   # fix axes positions before placing the captions under each pair
for (left, right), count in pair_axes:
    l, r = left.get_position(), right.get_position()
    fig.text((l.x0 + r.x1) / 2, l.y0 - 0.025, f"{count} errors", ha="center", va="top", fontsize=8, color=INK)
save(fig, "per_object_error.png")

In [ ]:
sorted(p.name for p in FIGURES.glob("*.png"))